
# Amazon Search Ranking — Week 6 V3
## FAST LLM Relevance Feature Experiment — Query-Batched

### Goal
Add an **LLM-generated query-product relevance score** to the leakage-safe Week 6 V2 Zero-Shot Adapter and test whether it improves cold-start ranking.

The LLM sees only information available for a new product:

- Query
- Product title
- Product description
- Brand
- Category

The LLM **never sees**:

- `__label__`
- review/rating counts
- clicks/purchases/interactions
- popularity
- LambdaMART/ListNet scores
- any historical signal

### Experimental design

1. Reuse the exact V2 FIXED query-level split:
   - 60% tuning queries
   - 40% held-out queries
2. Generate one LLM relevance score for every cold-start query-product pair.
3. Normalize LLM relevance **within each query**.
4. Tune only one fusion parameter on the tuning queries:

`Final Score = α × ZeroShotAdapter + (1 - α) × LLM relevance`

5. Report the official result only on held-out queries.

### Models compared

- BM25
- SBERT
- CLIP
- ZeroShotAdapter
- LLM-only
- ZeroShot + LLM
- LambdaMART
- ListNet

### Metrics

- NDCG@10
- NDCG@3
- Recall@10
- Recall@3
- MRR
- NDCG@10 improvement vs BM25

### Required uploads

Upload these two ZIP files together:

1. `amazon_week6_v1_coldstart_benchmark_results.zip`
2. `amazon_week6_v2_fixed_zero_shot_adapter_results.zip`

### Default local LLM

This notebook uses **Ollama + Llama 3.1 8B** locally in Colab.

For a lighter run, change:

`OLLAMA_MODEL = "llama3.1:8b"`

to:

`OLLAMA_MODEL = "llama3.2:3b"`

A GPU runtime is recommended.

### Final output

`amazon_week6_v3_llm_relevance_results.zip`


### FAST optimization
This version is designed for Colab speed:

- Scores **one query and all of its candidate products in one LLM call**
- Skips single-candidate queries because no ranking decision exists
- Defaults to `llama3.2:3b`
- Uses the same leakage-safe held-out evaluation as V2 FIXED

This reduces hundreds of pair-level LLM calls to roughly one call per multi-candidate query.


In [ ]:

# =========================
# 0. CONFIG
# =========================
import os, re, json, time, hashlib, shutil, zipfile, subprocess, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

ROOT = Path("/content/week6_v3")
UPLOAD_DIR = ROOT / "uploads"
EXTRACT_DIR = ROOT / "extracted"
OUT_DIR = ROOT / "outputs"
CACHE_DIR = ROOT / "cache"

for p in [ROOT, UPLOAD_DIR, EXTRACT_DIR, OUT_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

OLLAMA_MODEL = "llama3.2:3b"
TEMPERATURE = 0
MAX_RETRIES = 3
SAVE_EVERY = 10
FAILURE_RATE_LIMIT = 0.05

ALPHAS = np.round(np.linspace(0.0, 1.0, 41), 3)

print("Week 6 V3 config ready.")
print("Ollama model:", OLLAMA_MODEL)


In [ ]:

# =========================
# 1. UPLOAD V1 + V2 FIXED ZIP FILES
# =========================
from google.colab import files

uploaded = files.upload()

for name, content in uploaded.items():
    (UPLOAD_DIR / name).write_bytes(content)

zip_files = list(UPLOAD_DIR.glob("*.zip"))

if len(zip_files) < 2:
    print("WARNING: expected both V1 and V2 FIXED ZIPs. I will still inspect what was uploaded.")

print("\nUploaded:")
for p in zip_files:
    print(" -", p.name)


In [ ]:

# =========================
# 2. EXTRACT + LOAD
# =========================
for zpath in zip_files:
    dest = EXTRACT_DIR / zpath.stem
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as z:
        z.extractall(dest)

def find_file(name):
    hits = list(EXTRACT_DIR.rglob(name))
    return hits[0] if hits else None

v1_path = find_file("cold_start_subset.csv")
v2_path = find_file("zero_shot_scored_pairs.csv")

if v1_path is None:
    raise RuntimeError("V1 cold_start_subset.csv not found.")
if v2_path is None:
    raise RuntimeError("V2 FIXED zero_shot_scored_pairs.csv not found.")

v1 = pd.read_csv(v1_path)
v2 = pd.read_csv(v2_path)

meta_cols = [
    c for c in [
        "__query__", "__product__", "query", "title", "description",
        "brand", "category", "__label__"
    ]
    if c in v1.columns
]

meta = v1[meta_cols].drop_duplicates(["__query__", "__product__"]).copy()

v2_keep = [
    c for c in [
        "__query__", "__product__", "v2_split", "zero_shot_adapter_score",
        "bm25_score_qnorm", "sbert_similarity_qnorm", "clip_similarity_qnorm",
        "lambdamart_score", "listnet_score"
    ]
    if c in v2.columns
]

data = meta.merge(
    v2[v2_keep].drop_duplicates(["__query__", "__product__"]),
    on=["__query__", "__product__"],
    how="inner"
)

required = ["__query__", "__product__", "query", "title", "__label__", "v2_split", "zero_shot_adapter_score"]
missing = [c for c in required if c not in data.columns]
if missing:
    raise RuntimeError(f"Missing required columns: {missing}")

print("Merged V3 dataset:", data.shape)
print("Queries:", data["__query__"].nunique())
print("Products:", data["__product__"].nunique())
print("Tune queries:", data.loc[data["v2_split"]=="tune", "__query__"].nunique())
print("Holdout queries:", data.loc[data["v2_split"]=="holdout", "__query__"].nunique())

display(data.head(3))


In [ ]:
# =========================
# 3. INSTALL + START OLLAMA — FIXED FOR COLAB
# =========================
import os, shutil, subprocess, time, requests
from pathlib import Path

# 1) Install system dependency required by current Ollama installer.
print("Installing zstd...")
subprocess.run(
    ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq zstd curl"],
    check=True
)

# 2) Install Ollama only if the executable is missing.
ollama_bin = shutil.which("ollama")

if ollama_bin is None:
    print("Installing Ollama...")
    subprocess.run(
        ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
        check=True
    )
    ollama_bin = shutil.which("ollama")

# 3) Hard fail here if installation did not actually succeed.
if ollama_bin is None:
    possible = [
        "/usr/local/bin/ollama",
        "/usr/bin/ollama",
        "/bin/ollama",
    ]
    for p in possible:
        if Path(p).exists():
            ollama_bin = p
            break

if ollama_bin is None:
    raise RuntimeError(
        "Ollama installation did not produce an executable. "
        "Restart the runtime and rerun this cell."
    )

print("Ollama executable:", ollama_bin)

# 4) Check whether server is already running.
def ollama_alive():
    try:
        r = requests.get("http://127.0.0.1:11434/api/tags", timeout=3)
        return r.ok
    except Exception:
        return False

# 5) Start server if needed.
if not ollama_alive():
    log_path = "/tmp/ollama_v3.log"
    log_file = open(log_path, "w")

    proc = subprocess.Popen(
        [ollama_bin, "serve"],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"},
    )

    for i in range(45):
        if ollama_alive():
            break
        time.sleep(1)

if not ollama_alive():
    log_text = ""
    try:
        log_text = Path("/tmp/ollama_v3.log").read_text(errors="ignore")[-4000:]
    except Exception:
        pass

    raise RuntimeError(
        "Ollama server failed to start.\n\n"
        f"Last log output:\n{log_text}"
    )

print("Ollama server is running.")

# 6) Pull selected model.
print("Pulling model:", OLLAMA_MODEL)
subprocess.run(
    [ollama_bin, "pull", OLLAMA_MODEL],
    check=True
)

print("Model ready:", OLLAMA_MODEL)


## LLM scoring rubric

The prompt uses a fixed 0–100 scale:

- **90–100**: exact or near-exact product for the query
- **70–89**: strongly relevant; satisfies the main intent with minor differences
- **40–69**: partially relevant / plausible substitute
- **10–39**: weakly related
- **0–9**: unrelated or contradicts the query

The model is asked to judge **product-query relevance**, not product quality or popularity.


In [ ]:
# =========================
# 4. QUERY-BATCHED PROMPT + JSON PARSER
# =========================
import requests

PROMPT_VERSION = "week6_v3_query_batch_v2"

SYSTEM_RULES = """
You are an e-commerce search relevance judge.

For ONE search query, score EVERY candidate product independently for relevance.

Judge:
- product type match
- semantic intent match
- required attributes
- brand/model compatibility when explicitly requested

Do NOT use or infer:
- popularity
- reviews/ratings
- clicks/purchases/sales
- behavioral history

Do NOT invent missing facts.

Scoring:
90-100 = exact or near-exact match
70-89 = strongly relevant
40-69 = partially relevant / plausible substitute
10-39 = weakly related
0-9 = unrelated or contradictory

Return ONLY valid JSON:
{
  "results": [
    {
      "candidate_id": 0,
      "relevance_score": 0,
      "intent_match": false,
      "attribute_match": 0,
      "reason": "brief reason"
    }
  ]
}

Return exactly one result for every candidate_id.
""".strip()

def clean_text(x, limit):
    if pd.isna(x):
        return ""
    x = re.sub(r"\s+", " ", str(x)).strip()
    return x[:limit]

def build_query_prompt(query_text, g):
    blocks = []
    for cid, (_, row) in enumerate(g.reset_index(drop=True).iterrows()):
        title = clean_text(row.get("title", ""), 350)
        description = clean_text(row.get("description", ""), 500)
        brand = clean_text(row.get("brand", ""), 100)
        category = clean_text(row.get("category", ""), 120)

        blocks.append(
            f"""CANDIDATE {cid}
TITLE: {title}
DESCRIPTION: {description if description else "[missing]"}
BRAND: {brand if brand else "[missing]"}
CATEGORY: {category if category else "[missing]"}"""
        )

    return f"""{SYSTEM_RULES}

SEARCH QUERY:
{clean_text(query_text, 250)}

CANDIDATES:
{chr(10).join(chr(10) + b for b in blocks)}
""".strip()

def parse_batch_json(text, expected_n):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?", "", text, flags=re.I).strip()
    text = re.sub(r"```$", "", text).strip()

    try:
        obj = json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.S)
        if not m:
            raise ValueError("No JSON object found")
        obj = json.loads(m.group(0))

    results = obj.get("results", None)
    if not isinstance(results, list):
        raise ValueError("Missing results list")

    parsed = {}
    for item in results:
        cid = int(item["candidate_id"])
        score = int(np.clip(round(float(item["relevance_score"])), 0, 100))
        attr = int(np.clip(round(float(item.get("attribute_match", score))), 0, 100))
        intent = item.get("intent_match", False)
        if isinstance(intent, str):
            intent = intent.strip().lower() in ["true", "yes", "1"]

        parsed[cid] = {
            "relevance_score": score,
            "intent_match": bool(intent),
            "attribute_match": attr,
            "reason": clean_text(item.get("reason", ""), 160)
        }

    if set(parsed.keys()) != set(range(expected_n)):
        raise ValueError(
            f"Expected candidate ids 0..{expected_n-1}, got {sorted(parsed.keys())}"
        )

    return parsed

def call_ollama(prompt):
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {
            "temperature": TEMPERATURE,
            "seed": 42,
        }
    }

    r = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json=payload,
        timeout=180
    )
    r.raise_for_status()
    return r.json()["response"]

# Preview only
preview_query, preview_group = next(
    (q, g) for q, g in data.groupby("__query__") if len(g) >= 2
)
print(build_query_prompt(preview_group.iloc[0]["query"], preview_group)[:2500])

In [ ]:
# =========================
# 5. FAST QUERY-BATCHED LLM SCORING
# =========================
cache_path = CACHE_DIR / "llm_relevance_query_batch_cache.csv"

if cache_path.exists():
    cache = pd.read_csv(cache_path)
else:
    cache = pd.DataFrame(columns=[
        "__query__", "__product__", "llm_relevance_score",
        "llm_intent_match", "llm_attribute_match",
        "llm_reason", "llm_status", "llm_attempts",
        "llm_model", "prompt_version"
    ])

done_keys = set(
    zip(
        cache.get("__query__", pd.Series(dtype=str)).astype(str),
        cache.get("__product__", pd.Series(dtype=str)).astype(str)
    )
)

rows = cache.to_dict("records")

# Only multi-candidate queries matter for ranking.
multi_groups = [
    (q, g.copy())
    for q, g in data.groupby("__query__", sort=False)
    if len(g) >= 2
]

print("Multi-candidate queries to score:", len(multi_groups))
print("Total pair rows inside those queries:", sum(len(g) for _, g in multi_groups))

completed_queries = 0

for q, g in multi_groups:
    keys = [
        (str(row["__query__"]), str(row["__product__"]))
        for _, row in g.iterrows()
    ]

    if all(k in done_keys for k in keys):
        completed_queries += 1
        continue

    g_reset = g.reset_index(drop=True)
    query_text = g_reset.iloc[0]["query"]
    prompt = build_query_prompt(query_text, g_reset)

    parsed = None
    last_error = None
    attempts = 0

    for attempt in range(1, MAX_RETRIES + 1):
        attempts = attempt
        try:
            raw = call_ollama(prompt)
            parsed = parse_batch_json(raw, len(g_reset))
            break
        except Exception as e:
            last_error = repr(e)
            time.sleep(0.5 * attempt)

    if parsed is None:
        for cid, row in g_reset.iterrows():
            key = (str(row["__query__"]), str(row["__product__"]))
            result = {
                "__query__": row["__query__"],
                "__product__": row["__product__"],
                "llm_relevance_score": np.nan,
                "llm_intent_match": np.nan,
                "llm_attribute_match": np.nan,
                "llm_reason": "",
                "llm_status": f"FAILED_QUERY: {last_error}",
                "llm_attempts": attempts,
                "llm_model": OLLAMA_MODEL,
                "prompt_version": PROMPT_VERSION,
            }
            rows.append(result)
            done_keys.add(key)
    else:
        for cid, row in g_reset.iterrows():
            key = (str(row["__query__"]), str(row["__product__"]))
            item = parsed[cid]

            result = {
                "__query__": row["__query__"],
                "__product__": row["__product__"],
                "llm_relevance_score": item["relevance_score"],
                "llm_intent_match": item["intent_match"],
                "llm_attribute_match": item["attribute_match"],
                "llm_reason": item["reason"],
                "llm_status": "OK",
                "llm_attempts": attempts,
                "llm_model": OLLAMA_MODEL,
                "prompt_version": PROMPT_VERSION,
            }
            rows.append(result)
            done_keys.add(key)

    completed_queries += 1

    if completed_queries % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(cache_path, index=False)
        print(f"Scored {completed_queries}/{len(multi_groups)} multi-candidate queries")

# Single-candidate queries do not affect ranking metrics.
# Give them a neutral score instead of wasting LLM calls.
single = data.groupby("__query__").filter(lambda g: len(g) == 1)

for _, row in single.iterrows():
    key = (str(row["__query__"]), str(row["__product__"]))
    if key in done_keys:
        continue

    rows.append({
        "__query__": row["__query__"],
        "__product__": row["__product__"],
        "llm_relevance_score": 50,
        "llm_intent_match": np.nan,
        "llm_attribute_match": 50,
        "llm_reason": "single-candidate query; neutral score",
        "llm_status": "SKIPPED_SINGLE_CANDIDATE",
        "llm_attempts": 0,
        "llm_model": OLLAMA_MODEL,
        "prompt_version": PROMPT_VERSION,
    })
    done_keys.add(key)

cache = pd.DataFrame(rows)
cache.to_csv(cache_path, index=False)

rank_relevant_cache = cache[
    cache["llm_status"].isin(["OK", "FAILED_QUERY"])
].copy()

if len(rank_relevant_cache):
    failure_rate = (
        rank_relevant_cache["llm_status"] != "OK"
    ).mean()
else:
    failure_rate = 0.0

print("\nQuery-batched LLM scoring complete.")
print(cache["llm_status"].value_counts(dropna=False))
print("Failure rate on ranking-relevant rows:", f"{failure_rate:.2%}")

if failure_rate > FAILURE_RATE_LIMIT:
    raise RuntimeError(
        f"LLM failure rate {failure_rate:.2%} exceeds limit "
        f"{FAILURE_RATE_LIMIT:.0%}."
    )

In [ ]:

# =========================
# 6. MERGE LLM SCORE + DIAGNOSTICS
# =========================
llm_cols = [
    "__query__", "__product__", "llm_relevance_score",
    "llm_intent_match", "llm_attribute_match",
    "llm_reason", "llm_status"
]

data = data.merge(
    cache[llm_cols].drop_duplicates(["__query__", "__product__"]),
    on=["__query__", "__product__"],
    how="left"
)

data["llm_relevance_score"] = pd.to_numeric(
    data["llm_relevance_score"], errors="coerce"
).fillna(50.0)

def qnorm(series, qids):
    mn = series.groupby(qids).transform("min")
    mx = series.groupby(qids).transform("max")
    den = mx - mn
    return np.where(
        den.abs() > 1e-12,
        (series - mn) / den,
        0.5
    )

data["llm_relevance_qnorm"] = qnorm(
    data["llm_relevance_score"].astype(float),
    data["__query__"]
)

data["zero_shot_qnorm_v3"] = qnorm(
    pd.to_numeric(data["zero_shot_adapter_score"], errors="coerce").fillna(0.0),
    data["__query__"]
)

diag_rows = []
for split_name, part in [
    ("all", data),
    ("tune", data[data["v2_split"]=="tune"]),
    ("holdout", data[data["v2_split"]=="holdout"]),
]:
    multi = 0
    varying = 0

    for _, g in part.groupby("__query__"):
        if len(g) < 2:
            continue
        multi += 1
        if g["llm_relevance_score"].nunique() > 1:
            varying += 1

    diag_rows.append({
        "split": split_name,
        "multi_candidate_queries": multi,
        "queries_with_varying_llm_scores": varying,
        "varying_share": varying / max(multi, 1),
        "mean_llm_score": part["llm_relevance_score"].mean(),
        "std_llm_score": part["llm_relevance_score"].std(),
    })

llm_diag = pd.DataFrame(diag_rows)

display(llm_diag)
display(
    data[
        ["query", "title", "llm_relevance_score", "llm_intent_match", "llm_reason"]
    ].head(10)
)


In [ ]:

# =========================
# 7. LEAKAGE-SAFE RANKING METRICS
# =========================
def stable_unit(value, salt=""):
    h = hashlib.sha1((salt + str(value)).encode("utf-8")).hexdigest()
    return int(h[:12], 16) / float(16**12 - 1)

data["__tie_key__"] = data["__product__"].map(
    lambda p: stable_unit(p, "week6_v3_tie:")
)

def safe_order(scores, tie_keys):
    scores = np.asarray(scores, dtype=float)
    tie_keys = np.asarray(tie_keys, dtype=float)
    return np.lexsort((tie_keys, -scores))

def dcg_at_k(rels, k):
    rels = np.asarray(rels, dtype=float)[:k]
    if len(rels) == 0:
        return np.nan
    return float(np.sum(
        ((2.0 ** rels) - 1.0) /
        np.log2(np.arange(2, len(rels) + 2))
    ))

def ndcg_at_k(y, s, tie, k):
    y = np.asarray(y, dtype=float)
    order = safe_order(s, tie)
    ideal = np.sort(y)[::-1]
    denom = dcg_at_k(ideal, k)

    if not np.isfinite(denom) or denom <= 0:
        return np.nan

    return dcg_at_k(y[order], k) / denom

def recall_at_k(y, s, tie, k):
    y = np.asarray(y, dtype=float)
    rel = y > 0
    total = rel.sum()

    if total == 0:
        return np.nan

    order = safe_order(s, tie)[:k]
    return float(rel[order].sum() / total)

def reciprocal_rank(y, s, tie):
    y = np.asarray(y, dtype=float)
    order = safe_order(s, tie)
    hits = np.where(y[order] > 0)[0]
    return 0.0 if len(hits) == 0 else float(1.0 / (hits[0] + 1))

def evaluate(df, score_col, model, split):
    rows = []

    for q, g in df.groupby("__query__", sort=False):
        if len(g) < 2:
            continue

        y = g["__label__"].to_numpy(float)
        s = pd.to_numeric(g[score_col], errors="coerce").fillna(0.0).to_numpy(float)
        t = g["__tie_key__"].to_numpy(float)

        rows.append({
            "split": split,
            "model": model,
            "query": q,
            "n_candidates": len(g),
            "ndcg@10": ndcg_at_k(y, s, t, 10),
            "ndcg@3": ndcg_at_k(y, s, t, 3),
            "recall@10": recall_at_k(y, s, t, 10),
            "recall@3": recall_at_k(y, s, t, 3),
            "mrr": reciprocal_rank(y, s, t),
        })

    pq = pd.DataFrame(rows)

    return {
        "split": split,
        "model": model,
        "n_queries": int(len(pq)),
        "ndcg@10": float(pq["ndcg@10"].mean()) if len(pq) else np.nan,
        "ndcg@3": float(pq["ndcg@3"].mean()) if len(pq) else np.nan,
        "recall@10": float(pq["recall@10"].mean()) if len(pq) else np.nan,
        "recall@3": float(pq["recall@3"].mean()) if len(pq) else np.nan,
        "mrr": float(pq["mrr"].mean()) if len(pq) else np.nan,
    }, pq

print("Metric pipeline ready.")


In [ ]:

# =========================
# 8. TUNE ZERO-SHOT + LLM FUSION ON TUNE QUERIES ONLY
# =========================
tune = data[data["v2_split"] == "tune"].copy()
holdout = data[data["v2_split"] == "holdout"].copy()

fusion_rows = []

for alpha in ALPHAS:
    col = f"tmp_fusion_{alpha:.3f}"
    tune[col] = (
        alpha * tune["zero_shot_qnorm_v3"]
        + (1.0 - alpha) * tune["llm_relevance_qnorm"]
    )

    s, _ = evaluate(
        tune,
        score_col=col,
        model="ZeroShot+LLM",
        split="tune"
    )

    objective = (
        0.55 * s["ndcg@10"]
        + 0.25 * s["ndcg@3"]
        + 0.20 * s["mrr"]
    )

    fusion_rows.append({
        "alpha_zero_shot": alpha,
        "alpha_llm": 1.0 - alpha,
        "objective": objective,
        **{k: v for k, v in s.items() if k not in ["split", "model"]}
    })

fusion_search = pd.DataFrame(fusion_rows).sort_values(
    ["objective", "ndcg@10"],
    ascending=False
).reset_index(drop=True)

best_alpha = float(fusion_search.iloc[0]["alpha_zero_shot"])
best_llm_weight = 1.0 - best_alpha

print("Best fusion:")
print(f"ZeroShot weight = {best_alpha:.3f}")
print(f"LLM weight      = {best_llm_weight:.3f}")
display(fusion_search.head(10))

data["zero_shot_plus_llm_score"] = (
    best_alpha * data["zero_shot_qnorm_v3"]
    + best_llm_weight * data["llm_relevance_qnorm"]
)

tune = data[data["v2_split"] == "tune"].copy()
holdout = data[data["v2_split"] == "holdout"].copy()


In [ ]:

# =========================
# 9. OFFICIAL HELD-OUT V3 EVALUATION
# =========================
model_cols = {
    "BM25": "bm25_score_qnorm",
    "SBERT": "sbert_similarity_qnorm",
    "CLIP": "clip_similarity_qnorm",
    "ZeroShotAdapter": "zero_shot_adapter_score",
    "LLM-only": "llm_relevance_qnorm",
    "ZeroShot+LLM": "zero_shot_plus_llm_score",
    "LambdaMART": "lambdamart_score",
    "ListNet": "listnet_score",
}

model_cols = {
    m: c for m, c in model_cols.items()
    if c in data.columns
}

summaries = []
per_query_parts = []

for split_name, part in [
    ("tune", tune),
    ("holdout", holdout),
    ("full_cold_start", data),
]:
    for model, col in model_cols.items():
        s, pq = evaluate(part, col, model, split_name)
        summaries.append(s)
        if len(pq):
            per_query_parts.append(pq)

metrics = pd.DataFrame(summaries)
per_query = pd.concat(per_query_parts, ignore_index=True)

def add_bm25_improvement(df):
    out = df.copy()
    out["ndcg10_improvement_vs_bm25_pct"] = np.nan
    out["ndcg3_improvement_vs_bm25_pct"] = np.nan
    out["mrr_improvement_vs_bm25_pct"] = np.nan

    if "BM25" in set(out["model"]):
        b = out[out["model"] == "BM25"].iloc[0]

        for metric, target in [
            ("ndcg@10", "ndcg10_improvement_vs_bm25_pct"),
            ("ndcg@3", "ndcg3_improvement_vs_bm25_pct"),
            ("mrr", "mrr_improvement_vs_bm25_pct"),
        ]:
            if pd.notna(b[metric]) and b[metric] != 0:
                out[target] = (out[metric] - b[metric]) / b[metric] * 100

    return out

holdout_metrics = add_bm25_improvement(
    metrics[metrics["split"] == "holdout"].copy()
)
full_metrics = add_bm25_improvement(
    metrics[metrics["split"] == "full_cold_start"].copy()
)

print("OFFICIAL V3 HELD-OUT RESULTS")
display(
    holdout_metrics.sort_values("ndcg@10", ascending=False)
)

print("\nFULL COLD-START DIAGNOSTIC")
display(
    full_metrics.sort_values("ndcg@10", ascending=False)
)


In [ ]:

# =========================
# 10. V2 -> V3 DELTA
# =========================
def get_row(df, model):
    x = df[df["model"] == model]
    return None if x.empty else x.iloc[0]

v2_row = get_row(holdout_metrics, "ZeroShotAdapter")
v3_row = get_row(holdout_metrics, "ZeroShot+LLM")
bm25_row = get_row(holdout_metrics, "BM25")
lambda_row = get_row(holdout_metrics, "LambdaMART")

print("V2 -> V3 held-out comparison")

if v2_row is not None and v3_row is not None:
    print(f"V2 ZeroShot NDCG@10: {v2_row['ndcg@10']:.4f}")
    print(f"V3 ZeroShot+LLM NDCG@10: {v3_row['ndcg@10']:.4f}")
    print(f"Absolute delta: {v3_row['ndcg@10'] - v2_row['ndcg@10']:+.4f}")
    print(
        "Relative V3 improvement over V2:",
        f"{((v3_row['ndcg@10']/v2_row['ndcg@10'])-1)*100:+.2f}%"
    )

if v3_row is not None:
    print(
        "V3 improvement vs BM25:",
        f"{v3_row['ndcg10_improvement_vs_bm25_pct']:+.2f}%"
    )
    print(
        "15% Week 6 target achieved:",
        "YES" if v3_row["ndcg10_improvement_vs_bm25_pct"] >= 15 else "NO"
    )

if lambda_row is not None and v3_row is not None:
    print(
        "Gap to LambdaMART:",
        f"{v3_row['ndcg@10'] - lambda_row['ndcg@10']:+.4f} NDCG@10"
    )


In [ ]:

# =========================
# 11. ORDER-INDEPENDENCE TEST
# =========================
orig, _ = evaluate(
    holdout,
    "zero_shot_plus_llm_score",
    "ZeroShot+LLM",
    "holdout"
)

shuffled = holdout.sample(
    frac=1.0,
    random_state=2026
).reset_index(drop=True)

shuf, _ = evaluate(
    shuffled,
    "zero_shot_plus_llm_score",
    "ZeroShot+LLM",
    "holdout"
)

check = ["ndcg@10", "ndcg@3", "recall@10", "recall@3", "mrr"]
max_diff = max(abs(orig[c] - shuf[c]) for c in check)

print("Max metric difference after row shuffle:", max_diff)
assert max_diff < 1e-12, "ERROR: V3 evaluation depends on input row order."
print("PASS: V3 evaluation is row-order independent.")


In [ ]:

# =========================
# 12. LLM SCORE QUALITY DIAGNOSTICS
# =========================
score_by_label = (
    data.groupby("__label__", as_index=False)
        .agg(
            n=("llm_relevance_score", "size"),
            mean_llm_score=("llm_relevance_score", "mean"),
            median_llm_score=("llm_relevance_score", "median"),
            mean_zero_shot=("zero_shot_adapter_score", "mean")
        )
)

print("LLM score by relevance label:")
display(score_by_label)

example_cols = [
    "__query__", "__product__", "query", "title", "__label__",
    "llm_relevance_score", "llm_reason",
    "zero_shot_adapter_score", "zero_shot_plus_llm_score",
    "v2_split"
]

display(
    data[example_cols]
    .sort_values(["__query__", "llm_relevance_score"], ascending=[True, False])
    .head(20)
)


In [ ]:

# =========================
# 13. PLOTS
# =========================
p = holdout_metrics.dropna(subset=["ndcg@10"]).sort_values(
    "ndcg@10", ascending=False
)

plt.figure(figsize=(10, 5))
plt.bar(p["model"], p["ndcg@10"])
plt.ylabel("NDCG@10")
plt.xlabel("Model")
plt.title("Week 6 V3 — Held-Out Cold-Start NDCG@10")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "v3_holdout_ndcg10.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

p3 = holdout_metrics.dropna(subset=["ndcg@3"]).sort_values(
    "ndcg@3", ascending=False
)

plt.figure(figsize=(10, 5))
plt.bar(p3["model"], p3["ndcg@3"])
plt.ylabel("NDCG@3")
plt.xlabel("Model")
plt.title("Week 6 V3 — Held-Out Cold-Start NDCG@3")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "v3_holdout_ndcg3.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

curve = fusion_search.sort_values("alpha_zero_shot")

plt.figure(figsize=(8, 5))
plt.plot(curve["alpha_zero_shot"], curve["ndcg@10"], marker="o", markersize=3)
plt.xlabel("Alpha on V2 Zero-Shot Adapter")
plt.ylabel("Tune NDCG@10")
plt.title("Week 6 V3 — Zero-Shot / LLM Fusion Search")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "v3_fusion_alpha_curve.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(data["llm_relevance_score"].dropna(), bins=20)
plt.xlabel("LLM relevance score")
plt.ylabel("Count")
plt.title("Week 6 V3 — LLM Relevance Score Distribution")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "v3_llm_score_distribution.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()


In [ ]:

# =========================
# 14. SAVE V3 OUTPUTS
# =========================
cache.to_csv(
    OUT_DIR / "llm_relevance_scores.csv",
    index=False
)

llm_diag.to_csv(
    OUT_DIR / "llm_score_diagnostics.csv",
    index=False
)

score_by_label.to_csv(
    OUT_DIR / "llm_score_by_label_diagnostic.csv",
    index=False
)

fusion_search.to_csv(
    OUT_DIR / "llm_fusion_search.csv",
    index=False
)

holdout_metrics.to_csv(
    OUT_DIR / "v3_holdout_metrics.csv",
    index=False
)

full_metrics.to_csv(
    OUT_DIR / "v3_full_coldstart_metrics.csv",
    index=False
)

per_query.to_csv(
    OUT_DIR / "v3_per_query_metrics.csv",
    index=False
)

save_cols = [
    c for c in [
        "__query__", "__product__", "query", "title",
        "description", "brand", "category",
        "__label__", "v2_split",
        "zero_shot_adapter_score",
        "llm_relevance_score",
        "llm_relevance_qnorm",
        "llm_intent_match",
        "llm_attribute_match",
        "llm_reason",
        "zero_shot_plus_llm_score",
        "bm25_score_qnorm",
        "sbert_similarity_qnorm",
        "clip_similarity_qnorm",
        "lambdamart_score",
        "listnet_score"
    ]
    if c in data.columns
]

data[save_cols].to_csv(
    OUT_DIR / "v3_scored_pairs.csv",
    index=False
)

config = {
    "llm_model": OLLAMA_MODEL,
    "prompt_version": PROMPT_VERSION,
    "temperature": TEMPERATURE,
    "best_alpha_zero_shot": best_alpha,
    "best_alpha_llm": best_llm_weight,
    "tuning_metric": "0.55*NDCG@10 + 0.25*NDCG@3 + 0.20*MRR",
    "label_used_in_prompt": False,
    "history_used_in_prompt": False,
    "lambda_listnet_used_in_prompt": False,
    "row_order_independent": True,
}

with open(OUT_DIR / "v3_experiment_config.json", "w") as f:
    json.dump(config, f, indent=2)

lines = [
    "Amazon Search Ranking — Week 6 V3",
    "LLM Relevance Feature Experiment",
    "=" * 64,
    "",
    f"LLM model: {OLLAMA_MODEL}",
    f"Prompt version: {PROMPT_VERSION}",
    f"Best fusion: ZeroShot={best_alpha:.3f}, LLM={best_llm_weight:.3f}",
    "",
    "Official held-out results:"
]

for _, r in holdout_metrics.sort_values("ndcg@10", ascending=False).iterrows():
    imp = r.get("ndcg10_improvement_vs_bm25_pct", np.nan)
    imp_txt = "n/a" if pd.isna(imp) else f"{imp:+.2f}%"

    lines.append(
        f"- {r['model']}: "
        f"NDCG@10={r['ndcg@10']:.4f}, "
        f"NDCG@3={r['ndcg@3']:.4f}, "
        f"Recall@3={r['recall@3']:.4f}, "
        f"MRR={r['mrr']:.4f}, "
        f"vs BM25={imp_txt}"
    )

if v2_row is not None and v3_row is not None:
    lines += [
        "",
        f"V2 ZeroShot NDCG@10: {v2_row['ndcg@10']:.4f}",
        f"V3 ZeroShot+LLM NDCG@10: {v3_row['ndcg@10']:.4f}",
        f"V3 absolute delta vs V2: {v3_row['ndcg@10'] - v2_row['ndcg@10']:+.4f}",
        f"V3 vs BM25: {v3_row['ndcg10_improvement_vs_bm25_pct']:+.2f}%",
        "15% target achieved: "
        + ("YES" if v3_row["ndcg10_improvement_vs_bm25_pct"] >= 15 else "NO"),
    ]

summary_text = "\n".join(lines)

(OUT_DIR / "week6_v3_summary.txt").write_text(summary_text)

print(summary_text)


In [ ]:

# =========================
# 15. PACKAGE + DOWNLOAD
# =========================
zip_path = ROOT / "amazon_week6_v3_fast_llm_relevance_results.zip"

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.relative_to(OUT_DIR))

print("Created:", zip_path)
print("Size:", f"{zip_path.stat().st_size / 1024 / 1024:.2f} MB")

from google.colab import files
files.download(str(zip_path))



## How to interpret V3

Use **held-out** performance for the official conclusion.

### Strong V3 result
If `ZeroShot+LLM` improves clearly over `ZeroShotAdapter`, the LLM relevance signal is adding useful semantic reasoning for minimal-history products.

### Week 6 target
The target remains:

**Cold-start NDCG@10 ≥ 15% above BM25**

### If V3 still does not reach +15%
That is still a valid experimental result. V4 can then:

- compare V1 / V2 / V3 systematically,
- run ablations,
- optionally attempt contrastive fine-tuning,
- document contrastive learning as a stretch goal if the data is too small.
